# RetailMart Data Validation and Exploratory Analysis

This notebook documents the Python, Pandas, PostgreSQL/Neon, and KPI-analysis workflow used to support the AI-Assisted Cross-Functional Business Intelligence Dashboard.

**Important:** The notebook uses a synthetic dataset and does not contain database passwords or connection strings.

## 1. Analytical objectives

- Validate database connectivity and required RetailMart tables.
- Profile table availability, row counts, and missing values.
- Build the Sales Fact table.
- Calculate executive KPIs and monthly performance trends.
- Segment customers using RFM logic.
- Generate transparent rule-based management prompts.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

# Enable imports when the notebook is opened from the notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.bi_pipeline import (
    build_rule_based_insights,
    build_sales_fact,
    calculate_executive_kpis,
    create_postgres_engine,
    data_quality_profile,
    load_tables_from_postgres,
    monthly_sales_summary,
    rfm_segmentation,
    test_database_connection,
    validate_required_tables,
)

pd.set_option('display.max_columns', 30)
print(f'Repository root: {ROOT}')

## 2. Secure PostgreSQL/Neon connection

Set the environment variables in your local terminal before running this cell. Use Streamlit Secrets only for the deployed app. Do not paste passwords into the notebook.

In [ ]:
required_env = ['POSTGRES_HOST', 'POSTGRES_DATABASE', 'POSTGRES_USER', 'POSTGRES_PASSWORD']
missing = [name for name in required_env if not os.getenv(name)]

if missing:
    print('Connection not attempted. Set the following local environment variables first:')
    print(', '.join(missing))
else:
    engine = create_postgres_engine(
        host=os.environ['POSTGRES_HOST'],
        port=int(os.getenv('POSTGRES_PORT', '5432')),
        database=os.environ['POSTGRES_DATABASE'],
        user=os.environ['POSTGRES_USER'],
        password=os.environ['POSTGRES_PASSWORD'],
        sslmode=os.getenv('POSTGRES_SSLMODE', 'require'),
    )
    print(test_database_connection(engine))

## 3. Load mapped tables and validate required data

This cell loads only mapped project tables. Unavailable optional tables are reported as warnings rather than silently ignored.

In [ ]:
if 'engine' in globals():
    data, load_messages = load_tables_from_postgres(engine)
    for message in load_messages:
        print(f'{message.severity.upper()}: {message.message}')

    required_validation = validate_required_tables(data)
    display(required_validation)
else:
    print('Run the connection cell after setting local environment variables.')

## 4. Data-quality profile

The profile documents row counts, column counts, missing-cell rates, and duplicate-record checks after the data-cleaning process.

In [ ]:
if 'data' in globals():
    quality_profile = data_quality_profile(data)
    display(quality_profile.sort_values('table_key'))

## 5. Build Sales Fact and calculate executive KPIs

The Sales Fact combines valid order, order-item, product, brand, category, store, and region information. It is the central analytical structure for sales, profit, margin, trends, and customer segmentation.

In [ ]:
if 'data' in globals():
    sales_fact = build_sales_fact(data)
    print(f'Sales Fact rows: {len(sales_fact):,}')
    executive_kpis = calculate_executive_kpis(sales_fact)
    display(pd.DataFrame([executive_kpis]))

## 6. Monthly revenue and profit trend

In [ ]:
if 'sales_fact' in globals():
    monthly_summary = monthly_sales_summary(sales_fact)
    display(monthly_summary.tail(12))

## 7. RFM customer segmentation and CLV proxy

RFM means recency, frequency, and monetary value. The CLV measure is a proxy based on observed transactions, not a predictive lifetime-value model.

In [ ]:
if 'sales_fact' in globals():
    rfm = rfm_segmentation(sales_fact)
    segment_summary = (
        rfm.groupby('segment', as_index=False)
        .agg(customers=('cust_id', 'nunique'), monetary=('monetary', 'sum'), clv_proxy=('clv_proxy', 'sum'))
        .sort_values('monetary', ascending=False)
    )
    display(segment_summary)

## 8. Explainable rule-based AI-assisted insights

The dashboard uses advisory rules based on KPI thresholds and comparative performance. It does **not** use machine learning, deep learning, or generative AI.

In [ ]:
if 'sales_fact' in globals():
    for index, insight in enumerate(build_rule_based_insights(sales_fact), start=1):
        print(f'{index}. {insight}')

## 9. Academic interpretation

This notebook provides technical evidence for the project methodology. Findings must be interpreted as evidence of dashboard capability because the RetailMart data is synthetic. Any production implementation would require source-system reconciliation, formal KPI ownership, role-based access, refresh monitoring, and user-adoption evaluation.